In [4]:
import cv2
import glob
import io
import ipywidgets as widgets
import numpy as np
from pathlib import Path
from IPython.display import display


def _odd_kernel(value: int) -> int:
    value = max(1, int(value))
    if value % 2 == 0:
        value += 1
    return value


def _threshold_image(gray, method, adaptive_block, adaptive_c):
    if method == 'adaptive':
        block = _odd_kernel(adaptive_block)
        return cv2.adaptiveThreshold(
            gray,
            255,
            cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
            cv2.THRESH_BINARY,
            block,
            adaptive_c,
        )
    _, binary = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return binary


def _auto_invert(binary):
    white_ratio = cv2.countNonZero(binary) / float(binary.size)
    if white_ratio > 0.5:
        return cv2.bitwise_not(binary)
    return binary


def _contour_center(contour):
    moments = cv2.moments(contour)
    if moments['m00'] == 0:
        return None
    cx = int(moments['m10'] / moments['m00'])
    cy = int(moments['m01'] / moments['m00'])
    return cx, cy


def detect_blobs(
    path,
    min_radius,
    max_radius,
    blur,
    threshold_method,
    adaptive_block,
    adaptive_c,
    morph_kernel,
    morph_open_iter,
    morph_close_iter,
    auto_invert=True,
    invert=False,
):
    image = cv2.imread(str(path))
    if image is None:
        raise FileNotFoundError(f'Could not read image: {path}')

    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    blur_k = _odd_kernel(blur)
    blurred = cv2.GaussianBlur(gray, (blur_k, blur_k), 0)

    binary = _threshold_image(blurred, threshold_method, adaptive_block, adaptive_c)
    if auto_invert:
        binary = _auto_invert(binary)
    if invert:
        binary = cv2.bitwise_not(binary)

    morph_k = _odd_kernel(morph_kernel)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_k, morph_k))
    opened = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=morph_open_iter)
    if morph_close_iter > 0:
        cleaned = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=morph_close_iter)
    else:
        cleaned = opened

    contours, _ = cv2.findContours(cleaned, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    annotated = image.copy()
    kept = 0
    for contour in contours:
        area = cv2.contourArea(contour)
        if area <= 0:
            continue
        radius = np.sqrt(area / np.pi)
        if radius < min_radius or radius > max_radius:
            continue
        center = _contour_center(contour)
        if center is None:
            continue
        kept += 1
        cv2.circle(annotated, center, int(round(radius)), (0, 0, 255), 2)

    return kept, annotated, cleaned


In [5]:
image_files = sorted(glob.glob('Data/*'))
image_files = [
    path
    for path in image_files
    if Path(path).suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}
]
if not image_files:
    raise ValueError('No images found in Data/.')

first_image = cv2.imread(image_files[0])
if first_image is None:
    raise FileNotFoundError(f'Could not read image: {image_files[0]}')
first_height, first_width = first_image.shape[:2]
max_allowed = min(first_height, first_width) / 2.0

# UI
dropdown = widgets.Dropdown(
    options=image_files,
    value=image_files[0],
    description='Photo:',
)
min_radius = widgets.FloatSlider(
    value=2.0,
    min=1.0,
    max=max_allowed,
    step=0.5,
    description='Min radius:',
    readout_format='.1f',
)
max_radius = widgets.FloatSlider(
    value=max_allowed,
    min=1.0,
    max=max_allowed,
    step=0.5,
    description='Max radius:',
    readout_format='.1f',
)
blur = widgets.IntSlider(
    value=3,
    min=1,
    max=15,
    step=2,
    description='Blur (odd):',
)
threshold_method = widgets.Dropdown(
    options=['adaptive', 'otsu'],
    value='adaptive',
    description='Thresh:',
)
adaptive_block = widgets.IntSlider(
    value=31,
    min=3,
    max=61,
    step=2,
    description='Adapt block:',
)
adaptive_c = widgets.FloatSlider(
    value=4.0,
    min=-10.0,
    max=10.0,
    step=0.5,
    description='Adapt C:',
    readout_format='.1f',
)
morph_kernel = widgets.IntSlider(
    value=3,
    min=1,
    max=11,
    step=2,
    description='Morph k:',
)
morph_open_iter = widgets.IntSlider(
    value=1,
    min=0,
    max=5,
    step=1,
    description='Morph open:',
)
morph_close_iter = widgets.IntSlider(
    value=1,
    min=0,
    max=5,
    step=1,
    description='Morph close:',
)
auto_invert = widgets.Checkbox(
    value=True,
    description='Auto invert',
)
invert = widgets.Checkbox(
    value=False,
    description='Invert',
)
show_mask = widgets.Checkbox(
    value=True,
    description='Show mask',
)
label = widgets.Label()
image = widgets.Image(value=b'', format='png')

controls = widgets.GridBox(
    children=[
        dropdown,
        min_radius,
        max_radius,
        blur,
        threshold_method,
        adaptive_block,
        adaptive_c,
        morph_kernel,
        morph_open_iter,
        morph_close_iter,
        auto_invert,
        invert,
        show_mask,
    ],
    layout=widgets.Layout(
        grid_template_columns='repeat(2, minmax(200px, 1fr))',
        grid_gap='8px 16px',
    ),
)


def on_user_input(change):
    path = dropdown.value
    preview = cv2.imread(path)
    if preview is None:
        label.value = f'Could not read image: {path}'
        return
    height, width = preview.shape[:2]
    max_allowed = min(height, width) / 2.0
    if max_radius.max != max_allowed:
        max_radius.max = max_allowed
    if min_radius.max != max_allowed:
        min_radius.max = max_allowed
    if max_radius.value > max_allowed:
        max_radius.value = max_allowed
    if min_radius.value > max_radius.value:
        min_radius.value = max(1.0, max_radius.value - 1.0)

    count, annotated, mask = detect_blobs(
        path=path,
        min_radius=min_radius.value,
        max_radius=max_radius.value,
        blur=blur.value,
        threshold_method=threshold_method.value,
        adaptive_block=adaptive_block.value,
        adaptive_c=adaptive_c.value,
        morph_kernel=morph_kernel.value,
        morph_open_iter=morph_open_iter.value,
        morph_close_iter=morph_close_iter.value,
        auto_invert=auto_invert.value,
        invert=invert.value,
    )

    if show_mask.value:
        mask_bgr = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR)
        preview_image = np.hstack([annotated, mask_bgr])
        suffix = ' | left=annotated right=mask'
    else:
        preview_image = annotated
        suffix = ''

    label.value = (
        f'Objects detected: {count} | '
        f'min={min_radius.value:.1f} max={max_radius.value:.1f} '
        f'blur={blur.value} method={threshold_method.value}{suffix}'
    )
    _, encoded = cv2.imencode('.png', preview_image)
    image.value = encoded.tobytes()


for widget in [
    dropdown,
    min_radius,
    max_radius,
    blur,
    threshold_method,
    adaptive_block,
    adaptive_c,
    morph_kernel,
    morph_open_iter,
    morph_close_iter,
    auto_invert,
    invert,
    show_mask,
]:
    widget.observe(on_user_input)

on_user_input(None)

display(widgets.VBox([controls, label, image]))
